# Sparse, Dense, and Hybrid Retrieval

| Field | Value |
|---|---|
| Stage | Retrieval and reranking |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Sparse and dense branches fail differently. Evaluate each branch before fusing ranks, and never add hybrid complexity without a measured reason.

## 30-Second Summary

This notebook compares BM25 with a low-rank dense LSA representation on the shared eight-document corpus and seven answerable golden questions. Reciprocal-rank fusion combines the branches without pretending their raw scores share a scale.

## Why This Matters

Sparse retrieval rewards exact terms; dense representations can group correlated vocabulary but introduce model and compression error. Directly averaging incomparable scores can make a hybrid system harder to reason about.

## Scope

| Covers | Does not cover |
|---|---|
| BM25, low-rank dense retrieval, hit rate/MRR, reciprocal-rank fusion | Neural embedding download, ANN indexes, large-scale latency benchmark |


## Mental Model

```text
query -> BM25 rank ----\
                         reciprocal-rank fusion -> final rank
query -> dense LSA rank-/
```


In [1]:
import numpy as np
from rank_bm25 import BM25Okapi
from rag_101 import TfidfVectorizer, load_corpus, load_golden_questions
from rag_101.lexical import tokenize

documents = load_corpus()
questions = [item for item in load_golden_questions() if item.relevant_doc_ids]
document_tokens = [tokenize(document.content) for document in documents]
len(documents), len(questions)


(8, 7)

## How It Works

BM25 uses term frequency, document frequency, and length normalization. The dense branch projects TF-IDF vectors into a five-dimensional latent space with SVD; unlike a neural embedding it cannot understand unseen synonyms, but it clearly demonstrates dense compression. Reciprocal-rank fusion (RRF) uses positions rather than incompatible score magnitudes.


## Baseline

BM25 is the sparse baseline. It builds an inverted-term-style scorer over tokenized documents and returns exact-term-sensitive rankings.


In [2]:
bm25 = BM25Okapi(document_tokens)

def sparse_rank(query: str) -> list[str]:
    scores = bm25.get_scores(tokenize(query))
    order = sorted(range(len(documents)), key=lambda index: (-scores[index], documents[index].id))
    return [documents[index].id for index in order]

[(question.id, sparse_rank(question.question)[:2]) for question in questions]


[('q-retention', ['northstar-retention', 'northstar-access']),
 ('q-index-delete', ['northstar-indexing', 'northstar-access']),
 ('q-rollback', ['northstar-deployments', 'northstar-access']),
 ('q-p1', ['northstar-incidents', 'northstar-backups']),
 ('q-roles', ['northstar-access', 'northstar-indexing']),
 ('q-invoice', ['northstar-billing', 'northstar-access']),
 ('q-recovery', ['northstar-backups', 'northstar-billing'])]

## Technique Implementation

The dense branch fits TF-IDF once, performs truncated SVD, and compares projected query/document vectors by dot product. Five dimensions deliberately create a lossy representation so the experiment exposes rather than hides the branch trade-off.


In [3]:
vectorizer = TfidfVectorizer().fit(document.content for document in documents)
document_matrix = np.array(vectorizer.transform(document.content for document in documents))
_, _, right_vectors = np.linalg.svd(document_matrix, full_matrices=False)
latent_basis = right_vectors[:5].T
dense_documents = document_matrix @ latent_basis

def dense_rank(query: str) -> list[str]:
    query_vector = np.array(vectorizer.transform_one(query)) @ latent_basis
    scores = dense_documents @ query_vector
    order = sorted(range(len(documents)), key=lambda index: (-scores[index], documents[index].id))
    return [documents[index].id for index in order]

[(question.id, dense_rank(question.question)[:2]) for question in questions]


[('q-retention', ['northstar-retention', 'northstar-backups']),
 ('q-index-delete', ['northstar-indexing', 'northstar-incidents']),
 ('q-rollback', ['northstar-deployments', 'northstar-billing']),
 ('q-p1', ['northstar-backups', 'northstar-incidents']),
 ('q-roles', ['northstar-access', 'northstar-billing']),
 ('q-invoice', ['northstar-billing', 'northstar-access']),
 ('q-recovery', ['northstar-backups', 'northstar-retention'])]

## Controlled Experiment

We hold corpus, questions, cutoff, and labels fixed. RRF assigns each document `1 / (60 + rank)` from each branch. Hit rate@1 and MRR are computed from the first relevant document.


In [4]:
def rrf_rank(query: str, constant: int = 60, weights: tuple[float, float] = (1.0, 1.0)) -> list[str]:
    branch_ranks = (sparse_rank(query), dense_rank(query))
    scores = {document.id: 0.0 for document in documents}
    for weight, ranking in zip(weights, branch_ranks, strict=True):
        for rank, document_id in enumerate(ranking, start=1):
            scores[document_id] += weight / (constant + rank)
    return sorted(scores, key=lambda document_id: (-scores[document_id], document_id))

def evaluate(rank_function) -> dict[str, float]:
    hits, reciprocal_ranks = [], []
    for question in questions:
        ranking = rank_function(question.question)
        relevant = set(question.relevant_doc_ids)
        first = next((rank for rank, doc_id in enumerate(ranking, 1) if doc_id in relevant), None)
        hits.append(ranking[0] in relevant)
        reciprocal_ranks.append(1 / first if first else 0.0)
    return {"hit_rate@1": sum(hits) / len(hits), "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks)}

results = {
    "bm25": evaluate(sparse_rank),
    "dense_lsa_5d": evaluate(dense_rank),
    "equal_rrf": evaluate(rrf_rank),
    "sparse_weighted_rrf": evaluate(lambda query: rrf_rank(query, weights=(2.0, 1.0))),
}
results


{'bm25': {'hit_rate@1': 1.0, 'mrr': 1.0},
 'dense_lsa_5d': {'hit_rate@1': 0.8571428571428571, 'mrr': 0.9285714285714286},
 'equal_rrf': {'hit_rate@1': 0.8571428571428571, 'mrr': 0.9285714285714286},
 'sparse_weighted_rrf': {'hit_rate@1': 1.0, 'mrr': 1.0}}

## Evaluation

On this exact-term-heavy set, BM25 reaches **1.00 hit rate@1** while the deliberately compressed LSA branch reaches **0.857**. Equal-weight RRF also falls to **0.857** because the weak branch can overturn one correct top result. Weighting the validated sparse branch 2:1 restores **1.00**. Hybrid is not automatically safer; its weights need evidence.


In [5]:
assert results["bm25"]["hit_rate@1"] == 1.0
assert results["dense_lsa_5d"]["hit_rate@1"] == 6 / 7
assert results["equal_rrf"]["hit_rate@1"] == 6 / 7
assert results["sparse_weighted_rrf"]["hit_rate@1"] == 1.0
assert all(sorted(rank_function(question.question)) == sorted(document.id for document in documents)
           for rank_function in (sparse_rank, dense_rank, rrf_rank) for question in questions)
print("Sparse/dense/hybrid checks passed.")


Sparse/dense/hybrid checks passed.


## Decision Guide

| Query shape | Start with |
|---|---|
| IDs, error codes, exact names | Sparse/BM25 |
| Paraphrases and conceptual language | Evaluated neural dense model |
| Mixed traffic | Hybrid after branch-level measurement |
| Incomparable branch scores | Rank fusion or calibrated normalization |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Hybrid worse than best branch | Weak branch adds noise | Measure branches and tune/disable by segment |
| One branch dominates | Raw scores averaged | Use rank fusion or calibrated scores |
| Dense misses exact code | Representation smooths rare token | Keep sparse branch |
| Offline score is perfect | Golden set favors exact terms | Add paraphrase and hard-negative cases |


## Production Notes

### Observability
Log per-branch ranks/scores, overlap, fusion contribution, query segment, latency, and index/model version.

### Safety and Guardrails
Apply the same authorization filter to every branch before fusion.

### Latency and Cost
Parallel branches reduce wall time but increase total work; enforce per-branch timeouts and a deterministic fallback.


## Practice

Add two paraphrased questions with no distinctive source terms and compare the three methods before changing any weights.

## Recall

Toggle - Recall: Why not average BM25 and cosine scores directly?
Their ranges and meanings differ.

Toggle - Recall: What did hybrid add here?
Robustness to the weaker dense branch, not a quality improvement over BM25.

## Sources

- [BM25 paper](https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf)
- [Reciprocal Rank Fusion](https://dl.acm.org/doi/10.1145/1571941.1572114)
- Repository golden dataset

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the bounded shared-corpus experiment | Add neural embeddings and paraphrase-heavy labels |
